# ETL com Python - Dados de Leads da Internacional
Este notebook realiza o processo de ETL sobre os dados exportados do CRM da corretora Internacional.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Adiciona caminho raiz para importações
BASE_DIR = Path.cwd().resolve().parents[1] #subiu para a raiz

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR)) # Adicionou a raiz ao caminho de busca

# Importa
from Utils.io import save_csv

# Caminho base do projeto

raw_dir = BASE_DIR / "Data" / "RAW"
processed_dir = BASE_DIR / "Data" / "PROCESSED"
final_dir = BASE_DIR / "Data" / "FINAL"
full_simulated_dir = BASE_DIR / "Data" / "RAW" / "FULL_SIMULATED"


## 1. Leitura do CSV exportado do CRM

In [2]:
# Leitura
# Caminho raiz dos arquivos simulados
base_path = Path.cwd().parents[1] / 'Data' / 'RAW' / 'FULL_SIMULATED'

# Lista todos os arquivos CSV recursivamente
csv_files = list(base_path.glob('**/*.csv'))

print(f"Total de arquivos encontrados: {len(csv_files)}")



Total de arquivos encontrados: 107


In [3]:
# Carrega todos os CSVs e concatena em um único DataFrame
df_full = pd.concat([pd.read_csv(file, parse_dates=['data_cadastro']) for file in csv_files], ignore_index=True)


In [4]:
# Verifica o total de dados consolidados
print(f"Total de linhas consolidadas: {len(df_full):,}")

Total de linhas consolidadas: 267,448


In [5]:
# Salva o DataFrame consolidado em um arquivo CSV

from datetime import datetime

# Data de hoje para versão do arquivo
hoje = datetime.now().strftime('%Y_%m_%d')
nome_arquivo = f'leads_completo_{hoje}.csv'

# Salvar o arquivo com a data no nome
save_csv(df_full, processed_dir, nome_arquivo)


✅ Arquivo salvo com sucesso: D:\pythonProject\Projeto_Dados_Corretora\Data\PROCESSED\leads_completo_2025_04_21.csv


## 2. Tratamento e padronização de dados

In [6]:
df = df_full.copy()
df['dias_ate_1o_trade'] = pd.to_numeric(df['dias_ate_1o_trade'], errors='coerce')
df['valor_deposito'] = pd.to_numeric(df['valor_deposito'], errors='coerce')
df['foi_convertido'] = df['status_conversao'] == 'Convertido'
df['ano_mes_cadastro'] = df['data_cadastro'].dt.to_period('M').astype(str)
df.head()

,lead_id,origem,data_cadastro,status_conversao,dias_ate_1o_trade,valor_deposito,perfil,pais,foi_convertido,ano_mes_cadastro
0,210428,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Ukraine,False,2025-01
1,210429,E-mail Marketing,2025-01-08,Não Convertido,NaN,0.0,Agressivo,Burundi,False,2025-01
2,210430,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Korea,False,2025-01
3,210431,Orgânico,2025-01-10,Não Convertido,NaN,0.0,Conservador,Turks and Caicos Islands,False,2025-01
4,210432,YouTube Ads,2025-01-11,Não Convertido,NaN,0.0,Conservador,Luxembourg,False,2025-01


## 3. Geração de indicadores por canal

In [7]:
agg_canais = df.groupby('origem').agg({
    'lead_id': 'count',
    'foi_convertido': 'mean',
    'valor_deposito': 'mean'
}).rename(columns={
    'lead_id': 'total_leads',
    'foi_convertido': 'taxa_conversao',
    'valor_deposito': 'deposito_medio'
}).reset_index()

agg_canais

,origem,total_leads,taxa_conversao,deposito_medio
0,E-mail Marketing,31955,0.049006,247.121558
1,Facebook,26603,0.052701,260.913880
2,Google Ads,26830,0.050205,254.609698
3,Indicação,13319,0.050604,249.946479
4,Instagram,40234,0.049287,247.745199
5,Orgânico,48033,0.050465,248.873032
6,TikTok,37721,0.050078,256.297359
7,WhatsApp,21409,0.051053,259.500040
8,YouTube Ads,21344,0.051068,258.815710


## 4. Exportação dos dados tratados para uso no Power BI ou Streamlit

In [8]:

# Salva os arquivos 
df.to_csv(processed_dir / "leads_processados_para_pbi.csv", index=False)
agg_canais.to_csv(final_dir / "indicadores_por_canal.csv", index=False)